In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter


Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpxs6xd8_i".


In [2]:
%reload_ext nvcc4jupyter



Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpu_u27esz".


In [3]:
!nvidia-smi
!nvcc --version


Mon Nov  2 05:26:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!nvidia-smi

Mon Nov  2 05:27:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import os, subprocess
if not os.path.isdir('/content/P3-CUDA-Paralela'):
    subprocess.run(['git','clone','--branch','main',
                    'https://github.com/Jorge162017/P3-CUDA-Paralela.git','/content/P3-CUDA-Paralela'], check=True)
os.chdir('/content/P3-CUDA-Paralela')
subprocess.run(['git', 'checkout', 'main'], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
print(subprocess.check_output(['git','log','-1','--format=%H %an %s'], text=True))

fa1e80ec3f17b3a9f80e00a6329d4fb6292f8dd0 Giovanni Santos feat: implementar acumulador compartido por bloque



In [6]:
import os, glob, shutil, subprocess
if shutil.which('nvcc') is None:
    candidatos = ['/usr/local/cuda/bin/nvcc'] + sorted(glob.glob('/usr/local/cuda*/bin/nvcc'))
    encontrado = next((c for c in candidatos if os.path.exists(c)), None)
    assert encontrado, 'No se encontro nvcc: verifica que el entorno sea GPU (nvidia-smi) y que el toolkit CUDA este instalado'
    os.environ['PATH'] = os.path.dirname(encontrado) + os.pathsep + os.environ['PATH']
print('nvcc:', shutil.which('nvcc'))
print(subprocess.check_output(['nvcc', '--version'], text=True))

nvcc: /usr/local/cuda/bin/nvcc
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0



In [7]:
import subprocess
os.makedirs('build', exist_ok=True)
cmd = ['nvcc','-I.','-O2','-std=c++17','--fmad=false','src/houghShared.cu','common/pgm.cpp','-o','build/hough_shared']
comp = subprocess.run(cmd, text=True, capture_output=True)
print(comp.stdout, comp.stderr)
assert comp.returncode == 0, 'La compilacion fallo'

In [8]:
import tempfile
from pathlib import Path
log  = 'Variante: global + compartida; tablas trigonometricas aun en memoria GLOBAL (sin memoria constante).\n'
log += 'Commit: ' + subprocess.check_output(['git','rev-parse','HEAD'], text=True)
log += 'Compilacion: ' + ' '.join(cmd) + '\n\n'
log += subprocess.check_output(['nvidia-smi'], text=True) + subprocess.check_output(['nvcc','--version'], text=True) + '\n'

def run_case(name, path, must_pass=True):
    global log
    r = subprocess.run(['./build/hough_shared', str(path)], text=True, capture_output=True)
    log += f'{name} (codigo de salida {r.returncode})\n' + r.stdout + r.stderr + '\n'
    print(f'{name}: codigo {r.returncode} \n', r.stdout, r.stderr)
    return r.returncode

results = {}
results['runway.pgm (original)'] = run_case('runway.pgm (original)', 'data/runway.pgm')
with tempfile.TemporaryDirectory() as d:
    casos = [('21 pixeles blancos (7x3, 1 bloque parcial)', 7, 3, 255),
             ('21 pixeles negros (7x3)', 7, 3, 0),
             ('300 pixeles blancos (20x15, 2 bloques, ultimo parcial)', 20, 15, 255)]
    for name, wd, ht, v in casos:
        p = Path(d)/'t.pgm'
        p.write_bytes(f'P5\n{wd} {ht}\n255\n'.encode() + bytes([v])*(wd*ht))
        results[name] = run_case(name, p)
for args in [[], ['archivo-inexistente.pgm']]:
    r = subprocess.run(['./build/hough_shared', *args], text=True, capture_output=True)
    log += f'argumentos invalidos {args} (codigo {r.returncode}) ===\n' + r.stdout + r.stderr + '\n'
    results[f'invalido {args}'] = 0 if r.returncode != 0 else 1   # debe fallar
log += ' Resumen\n' + '\n'.join(f'{k}: {v}' for k, v in results.items()) + '\n'
Path('results').mkdir(exist_ok=True)
Path('results/compartida-validacion.txt').write_text(log)
print('Resumen:', results)

runway.pgm (original): codigo 0 
 Variante: global + compartida (tablas aun en memoria global)
Imagen: 800 x 600; acumulador: 9000 bins
GPU: Tesla T4
Memoria compartida por bloque: requerida 36000 bytes, limite GPU 49152 bytes
Bloques: 1875; hilos por bloque: 256; pixeles en ultimo bloque: 256
VALIDACION OK: CPU y GPU coinciden en todos los bins.
 
21 pixeles blancos (7x3, 1 bloque parcial): codigo 0 
 Variante: global + compartida (tablas aun en memoria global)
Imagen: 7 x 3; acumulador: 9000 bins
GPU: Tesla T4
Memoria compartida por bloque: requerida 36000 bytes, limite GPU 49152 bytes
Bloques: 1; hilos por bloque: 256; pixeles en ultimo bloque: 21
VALIDACION OK: CPU y GPU coinciden en todos los bins.
 
21 pixeles negros (7x3): codigo 0 
 Variante: global + compartida (tablas aun en memoria global)
Imagen: 7 x 3; acumulador: 9000 bins
GPU: Tesla T4
Memoria compartida por bloque: requerida 36000 bytes, limite GPU 49152 bytes
Bloques: 1; hilos por bloque: 256; pixeles en ultimo bloque:

In [9]:
from google.colab import files
files.download('results/compartida-validacion.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>